In [5]:
import torch
import sys

sys.path.insert(0, "..")
from testbed.models import Idefics
import config

device = torch.device("cpu")
lmm = Idefics(
    config.idefics_9b_path,
    torch_dtype=torch.float16,
).to(device)

Loading checkpoint shards: 100%|██████████| 19/19 [00:06<00:00,  3.08it/s]


In [10]:
lmm.model.lm_head(torch.randn((1, 4096), dtype=torch.float16))

tensor([[-1.0000, -1.0654,  0.9595,  ..., -0.9468,  0.6924, -0.1333]],
       dtype=torch.float16, grad_fn=<CatBackward0>)

In [2]:
import enum
from functools import partial
from typing import List, Callable, Dict, Union
import torch
from torch import nn
import re
from types import MethodType
from testbed.models.model_base import HookType


class BaseHookEncoder(nn.Module):
    def __init__(self):
        super().__init__()

    def register_hooks(
        self,
        lmm,
        register_fn_name: str,
        targets: List[Union[str, HookType]],
        hooks: Dict[str, Callable],
    ):
        handles = dict()

        for target, (name, hook_fn) in zip(targets, hooks.items()):
            if hook_fn is not None and not "record" in name:
                handles[name] = getattr(lmm, register_fn_name)(
                    target, hook_fn, use_regex=True
                )

        # all record hooks should be called after shift hooks
        for target, (name, hook_fn) in zip(targets, hooks.items()):
            if hook_fn is not None and "record" in name:
                handles[name] = getattr(lmm, register_fn_name)(
                    target, hook_fn, use_regex=True
                )

        return handles

    def do_shift(self, hidden_states, shift: torch.Tensor):
        if shift.dim() < 2:
            shift = shift.unsqueeze(0)
        if shift.dim() < 3:
            shift = shift.unsqueeze(0)

        shifted_states = hidden_states + shift

        return shifted_states

    def record_hook(self, m, inputs, outputs, module_name, record_varname, **kwargs):
        layer_idx = int(re.findall(r"\d+", module_name)[0])
        if not isinstance(outputs, tuple):
            outputs = (outputs,)
        hidden_states, *_ = outputs
        getattr(self, record_varname)[layer_idx] = hidden_states


def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


# Copied from transformers.models.mixtral.modeling_mixtral.apply_rotary_pos_emb
def apply_rotary_pos_emb(q, k, cos, sin, position_ids, unsqueeze_dim=1):
    """Applies Rotary Position Embedding to the query and key tensors.

    Args:
        q (`torch.Tensor`): The query tensor.
        k (`torch.Tensor`): The key tensor.
        cos (`torch.Tensor`): The cosine part of the rotary embedding.
        sin (`torch.Tensor`): The sine part of the rotary embedding.
        position_ids (`torch.Tensor`):
            The position indices of the tokens corresponding to the query and key tensors. For example, this can be
            used to pass offsetted position ids when working with a KV-cache.
        unsqueeze_dim (`int`, *optional*, defaults to 1):
            The 'unsqueeze_dim' argument specifies the dimension along which to unsqueeze cos[position_ids] and
            sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note
            that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and
            k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes
            cos[position_ids] and sin[position_ids] broadcastable to the shapes of q and k. Similarly, if q and k have
            the shape [batch_size, seq_len, heads, head_dim], then set unsqueeze_dim=2.
    Returns:
        `tuple(torch.Tensor)` comprising of the query and key tensors rotated using the Rotary Position Embedding.
    """
    cos = cos[position_ids].unsqueeze(unsqueeze_dim)
    sin = sin[position_ids].unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class ShiftStrategy(enum.IntFlag):
    USE_VECTOR_IMPL = 1
    USE_LORA_IMPL = 2
    USE_STRANGE_ATTN_IMPL = 4
    RECORD_HIDDEN_STATES = 8
    FROZEN = 16


class AttnFFNShift(BaseHookEncoder):
    def __init__(
        self,
        lmm_hidden_dim,
        lmm_layers,
        attn_strategy: ShiftStrategy = None,
        ffn_strategy: ShiftStrategy = None,
        **kwargs,
    ):
        super().__init__()

        def parse_strategy(prefix, strategy):
            params = []
            setattr(self, f"{prefix}_strategy", strategy)

            def load_state_dict_post_hook(module, incompatible_keys):
                for p in params:
                    p.requires_grad_(ShiftStrategy.FROZEN not in strategy)

            if (
                bin(
                    strategy
                    & (
                        ShiftStrategy.USE_VECTOR_IMPL
                        | ShiftStrategy.USE_LORA_IMPL
                        | ShiftStrategy.USE_STRANGE_ATTN_IMPL
                    )
                ).count("1")
                > 1
            ):
                raise ValueError(
                    "The shift implementation methods are mutually exclusive."
                )

            def add_attr(name, value):
                setattr(self, name, value)
                params.append(getattr(self, name))

            if ShiftStrategy.USE_VECTOR_IMPL in strategy:
                add_attr(
                    f"{prefix}_shift",
                    torch.nn.Parameter(
                        torch.empty(lmm_layers, lmm_hidden_dim).normal_(
                            mean=0.0, std=0.001
                        )
                    ),
                )
            elif ShiftStrategy.USE_LORA_IMPL in strategy:
                r = kwargs.get("r", 8)
                add_attr(
                    f"{prefix}_lora_A",
                    torch.nn.Parameter(torch.randn(lmm_layers, lmm_hidden_dim, r)),
                )
                add_attr(
                    f"{prefix}_lora_B",
                    torch.nn.Parameter(torch.zeros(lmm_layers, r, lmm_hidden_dim)),
                )
            elif ShiftStrategy.USE_STRANGE_ATTN_IMPL in strategy:
                if prefix != "attn":
                    raise ValueError(
                        "ShiftStrategy.USE_STRANGE_ATTN_IMPL can only be used in attn strategy"
                    )
                add_attr("attn_k_linear", nn.Linear(lmm_hidden_dim, lmm_hidden_dim))
                add_attr("attn_q_linear", nn.Linear(lmm_hidden_dim, lmm_hidden_dim))
                # it will replace regular attn forward in register hooks method

            if ShiftStrategy.RECORD_HIDDEN_STATES in strategy:
                setattr(
                    self, f"{prefix}_hidden_states", [[] for _ in range(lmm_layers)]
                )

            self.register_load_state_dict_post_hook(load_state_dict_post_hook)

        if attn_strategy is not None:
            parse_strategy("attn", attn_strategy)
        if ffn_strategy is not None:
            parse_strategy("ffn", ffn_strategy)

    def attn_shift_params(self):
        for n, p in self.named_parameters():
            if n.startswith("attn"):
                yield n, p

    def ffn_shift_params(self):
        for n, p in self.named_parameters():
            if n.startswith("ffn"):
                yield n, p

    def freeze_attn_shift(self):
        for name, params in self.attn_shift_params():
            params.requires_grad_(False)

    def unfreeze_attn_shift(self):
        for name, params in self.attn_shift_params():
            params.requires_grad_(True)

    def freeze_ffn_shift(self):
        for name, params in self.ffn_shift_params():
            params.requires_grad_(False)

    def unfreeze_ffn_shift(self):
        for name, params in self.ffn_shift_params():
            params.requires_grad_(True)

    def new_attn_forward(
        self,
        hidden_states: torch.Tensor,
        key_value_states=None,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions: bool = False,
        use_cache: bool = False,
        module_name=None,
        shift_encoder=None,
    ):
        layer_idx = int(re.findall(r"\d+", module_name)[0])
        # if key_value_states are provided this layer is used as a cross-attention layer
        is_cross_attention = self.is_cross_attention or key_value_states is not None

        bsz, q_len, _ = hidden_states.size()

        query_states = (
            self.q_proj(hidden_states)
            .view(bsz, q_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )
        if not is_cross_attention:
            key_states = (
                self.k_proj(hidden_states)
                .view(bsz, q_len, self.num_heads, self.head_dim)
                .transpose(1, 2)
            )
            value_states = (
                self.v_proj(hidden_states)
                .view(bsz, q_len, self.num_heads, self.head_dim)
                .transpose(1, 2)
            )
        else:
            _, kv_len, _ = (
                key_value_states.size()
            )  # Note that, in this case, `kv_len` == `kv_seq_len`
            key_states = (
                self.k_proj(key_value_states)
                .view(bsz, kv_len, self.num_heads, self.head_dim)
                .transpose(1, 2)
            )
            value_states = (
                self.v_proj(key_value_states)
                .view(bsz, kv_len, self.num_heads, self.head_dim)
                .transpose(1, 2)
            )

        kv_seq_len = key_states.shape[-2]
        if past_key_value is not None:
            kv_seq_len += past_key_value[0].shape[-2]
        if not is_cross_attention:
            cos, sin = self.rotary_emb(value_states, seq_len=max(kv_seq_len, q_len))
            query_states, key_states = apply_rotary_pos_emb(
                query_states, key_states, cos, sin, position_ids
            )
        # [bsz, nh, t, hd]

        if past_key_value is not None:
            # reuse k, v, self_attention
            key_states = torch.cat([past_key_value[0], key_states], dim=2)
            value_states = torch.cat([past_key_value[1], value_states], dim=2)

        past_key_value = (key_states, value_states) if use_cache else None

        if self.qk_layer_norms:
            query_states = self.q_layer_norm(query_states)
            key_states = self.k_layer_norm(key_states)

        if attention_mask is not None:
            if attention_mask.size() != (bsz, 1, q_len, kv_seq_len):
                raise ValueError(
                    f"Attention mask should be of size {(bsz, 1, q_len, kv_seq_len)}, but is {attention_mask.size()}"
                )

        # SDPA with memory-efficient backend is currently (torch==2.1.2) bugged with non-contiguous inputs with custom attn_mask,
        # Reference: https://github.com/pytorch/pytorch/issues/112577.
        if query_states.device.type == "cuda" and attention_mask is not None:
            query_states = query_states.contiguous()
            key_states = key_states.contiguous()
            value_states = value_states.contiguous()

        # We dispatch to SDPA's Flash Attention or Efficient kernels via this `is_causal` if statement instead of an inline conditional assignment
        # in SDPA to support both torch.compile's dynamic shapes and full graph options. An inline conditional prevents dynamic shapes from compiling.
        # The q_len > 1 is necessary to match with AttentionMaskConverter.to_causal_4d that does not create a causal mask in case q_len == 1.
        is_causal = (
            True if self.is_causal and attention_mask is None and q_len > 1 else False
        )

        attn_output = torch.nn.functional.scaled_dot_product_attention(
            query_states,
            key_states,
            value_states,
            attn_mask=attention_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        if attn_output.size() != (bsz, self.num_heads, q_len, self.head_dim):
            raise ValueError(
                f"`attn_output` should be of size {(bsz, self.num_heads, q_len, self.head_dim)}, but is"
                f" {attn_output.size()}"
            )

        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.reshape(bsz, q_len, self.hidden_size)

        attn_output = self.o_proj(attn_output)

        attn_weights = None

        return attn_output, attn_weights, past_key_value

    def register_shift_hooks(self, lmm, **model_inputs):
        self_attn_layers = r"model\.layers\.\d+\.self\_attn$"
        mlp_layers = r"model\.layers\.\d+\.mlp$"

        if (
            hasattr(self, "attn_strategy")
            and ShiftStrategy.USE_STRANGE_ATTN_IMPL in self.attn_strategy
        ):
            lmm.replace_module_method(
                self_attn_layers,
                "forward",
                MethodType(partial(AttnFFNShift.new_attn_forward, shift_encoder=self)),
            )

        return self.register_hooks(
            lmm,
            "register_forward_hook",
            [
                self_attn_layers,
                mlp_layers,
            ],
            {
                "attn_hook": (
                    self._shift_hook("attn")
                    if hasattr(self, "attn_lora_A") or hasattr(self, "attn_shift")
                    else None
                ),
                "ffn_hook": (
                    self._shift_hook("ffn")
                    if hasattr(self, "ffn_lora_A") or hasattr(self, "ffn_shift")
                    else None
                ),
            },
        )

    def register_record_hooks(self, lmm, **model_inputs):
        self_attn_layers = r"model\.layers\.\d+\.self\_attn$"
        mlp_layers = r"model\.layers\.\d+\.mlp$"

        return self.register_hooks(
            lmm,
            "register_forward_hook",
            [
                self_attn_layers,
                mlp_layers,
            ],
            {
                "attn_record_hook": (
                    partial(self.record_hook, record_varname="attn_hidden_states")
                    if hasattr(self, "attn_hidden_states")
                    else None
                ),
                "ffn_record_hook": (
                    partial(self.record_hook, record_varname="ffn_hidden_states")
                    if hasattr(self, "ffn_hidden_states")
                    else None
                ),
            },
        )

    def _shift_hook(self, prefix):
        def hook(m, inputs, outputs, module_name, **kwargs):
            layer_idx = int(re.findall(r"\d+", module_name)[0])
            lora_A = getattr(self, f"{prefix}_lora_A", None)
            lora_B = getattr(self, f"{prefix}_lora_B", None)
            shift = getattr(self, f"{prefix}_shift", None)

            if isinstance(outputs, tuple):
                hidden_states, *rest = outputs

            if lora_A is not None and lora_B is not None:
                shift = (hidden_states @ lora_A[layer_idx]) @ lora_B[layer_idx]
            else:
                shift = shift[layer_idx]
            shifted_states = self.do_shift(hidden_states, shift)

            if isinstance(outputs, tuple):
                return (shifted_states, *rest)
            else:
                return shifted_states

        return hook

In [6]:
import pytorch_lightning as pl
import datasets
from torch.utils.data import (
    DistributedSampler,
    BatchSampler,
    SequentialSampler,
    RandomSampler,
)
import os
import sys
from pathlib import Path

from testbed.data import prepare_caption_input, prepare_dataloader, prepare_vqa_input
import config
import exp_settings as setting


class ICVDataModule(pl.LightningDataModule):

    def __init__(self, lmm) -> None:
        super().__init__()
        self.lmm = lmm
        if setting.dataset == "vqav2":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "vqav2"),
                split="train",
                data_dir=config.vqav2_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        elif setting.dataset == "ok_vqa":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "ok_vqa"),
                split="train",
                data_dir=config.ok_vqa_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        elif setting.dataset == "coco_cap":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "coco"),
                split="train",
                data_dir=config.karpathy_coco_caption_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        self.query_set = self.dataset.select(
            range(setting.num_query_samples)
        )

    def collate_fn(self, batch):
        """
        Split batch into full context, in-context examples, query and answer, and process them into model inputs.
        """
        if setting.dataset in ["vqav2", "ok_vqa"]:
            context, images = prepare_vqa_input(
                batch, instruction=setting.vqa_instruction
            )
            # we use the first answer as grounding truth
            answers = [item[-1]["answers"][0]["answer"] for item in batch]
        elif setting.dataset == "caption":
            context, images = prepare_caption_input(
                batch, instruction=setting.caption_instruction
            )
            answers = [item[-1]["sentences_raw"][0] for item in batch]

        # the last two items (take vqa as an example):
        # [
        #   { "role" : "question"
        #     "content" :  ... },
        #   { "role" : "short answer" }
        # ]
        ice_texts = self.lmm.apply_prompt_template([ctx[:-2] for ctx in context])
        query_texts = self.lmm.apply_prompt_template([ctx[-2:] for ctx in context])

        return {
            "ice_texts": ice_texts,
            "query_texts": query_texts,
            "answers": answers,
            "images": images,
        }

    def train_dataloader(self):
        if self.trainer and self.trainer.world_size > 1:
            samplers = [
                BatchSampler(
                    DistributedSampler(self.dataset, shuffle=False),
                    batch_size=setting.num_shot,
                    drop_last=True,
                ),
                DistributedSampler(self.query_set, shuffle=False),
            ]
        else:
            samplers = [
                BatchSampler(
                    SequentialSampler(self.dataset), batch_size=1, drop_last=True
                ),
                SequentialSampler(self.query_set),
            ]

        return prepare_dataloader(
            [self.dataset, self.query_set],
            2,
            num_per_dataset=[1, 1],
            collate_fn=self.collate_fn,
            samplers=samplers,
            # num_workers=1,
            pin_memory=True,
            shuffle=False,
        )

datamodule = ICVDataModule(lmm)
dataloader = datamodule.train_dataloader()

In [23]:
import enum
from functools import reduce
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
import torch.optim as optim
from deepspeed.ops.adam import DeepSpeedCPUAdam
from transformers import get_cosine_schedule_with_warmup

import exp_settings as setting

# from shift_encoder import AttnFFNShift, ShiftStrategy


class Stratety(enum.IntFlag):
    LAYER_WISE_KL_DIV = 1
    LAYER_WISE_MSE = 2
    LOGITS_KL_DIV = 4
    LM_LOSS = 8
    ALTERNATE_TRAINING = 16
    LAYER_WISE_COS_SIM = 32

    def has_layer_wise(self):
        try:
            self.layer_wise_strategy()
            return True
        except ValueError:
            return False

    def validate(self):
        layer_wise_loss = [
            Stratety.LAYER_WISE_KL_DIV,
            Stratety.LAYER_WISE_MSE,
            Stratety.LAYER_WISE_COS_SIM,
        ]

        if bin(self & reduce(lambda x, y: x | y, layer_wise_loss)).count("1") > 1:
            raise ValueError(
                f"{[e.name for e in layer_wise_loss]} are mutually exclusive."
            )

        if Stratety.ALTERNATE_TRAINING in self:
            if not (
                self.has_layer_wise()
                and self & (Stratety.LM_LOSS | Stratety.LOGITS_KL_DIV)
            ):
                raise ValueError(
                    "Strategy should contain a layer-wise loss and a task specific loss"
                    "if alternate training is enabled."
                )

    def layer_wise_strategy(self):
        if Stratety.LAYER_WISE_KL_DIV in self:
            return "kl_loss"
        elif Stratety.LAYER_WISE_MSE in self:
            return "mse_loss"
        elif Stratety.LAYER_WISE_COS_SIM in self:
            return "cos_sim"
        else:
            raise ValueError("None of layer wise loss strategy is enabled")




class ShiftModel(pl.LightningModule):
    def __init__(self, lmm, shift_encoder, strategy: Stratety) -> None:
        super().__init__()
        self.lmm = lmm
        self.lmm.requires_grad_(False)
        self.shift_encoder = shift_encoder
        strategy.validate()
        self.strategy = strategy

    def generate_label_mask(self, inputs, num_separator, keep_bos=False):
        """
        Generates label mask which masks tokens before num_separator pad_tokens from given inputs.
        """
        input_ids = inputs["input_ids"]
        batch_size, seq_len = input_ids.shape
        pad_mask = input_ids == self.lmm.processor.tokenizer.pad_token_id
        non_pad_mask = ~pad_mask
        label_mask = torch.zeros_like(input_ids, dtype=torch.bool)
        if self.lmm.processor.tokenizer.padding_side == "left":
            bos_position = non_pad_mask.int().argmax(dim=1)

        for i in range(batch_size):
            seq_pad_positions = pad_mask[i].nonzero(as_tuple=False).squeeze(-1)

            if self.lmm.processor.tokenizer.padding_side == "left":
                seq_pad_positions = seq_pad_positions[
                    seq_pad_positions > bos_position[i]
                ]

            num_pads = len(seq_pad_positions)
            if num_pads < num_separator:
                raise ValueError(
                    f"Sequence {i} has fewer pad tokens ({num_pads}) than num_separator ({num_separator})"
                )

            sep_position = seq_pad_positions[num_separator - 1].item()
            label_mask[i, sep_position + 1 :] = True

        label_mask = label_mask & non_pad_mask
        if keep_bos:
            label_mask[torch.arange(batch_size, device=self.device), bos_position] = (
                True
            )

        return label_mask

    def remove_hooks(self, hooks):
        # remove all hooks
        for name, handles in hooks.items():
            if isinstance(handles, list):
                for handle in handles:
                    handle.remove()
            else:
                handles.remove()

    def get_hidden_states(self, query_label_mask):
        """
        Apply query_label_mask to extract query parts from hidden states (shape: num_layer * [batch_size, seq_len, d_model]),
        and convert to batch_size * [num_layer, query_part_len, d_model].
        """
        hidden_states_dict = {}

        for name, attr in vars(self.shift_encoder).items():
            if "hidden_states" in name:
                # [num_layer, batch_size, seq_len, d_model] -> [batch_size, num_layer, seq_len, d_model]
                hidden_states = torch.stack(attr).transpose(0, 1)
                batch_size, num_layer, seq_len, d_model = hidden_states.shape
                hidden_states_dict[name] = [
                    hs.masked_select(mask[None, :, None]).view(num_layer, -1, d_model)
                    for hs, mask in zip(hidden_states, query_label_mask)
                ]

        if not hidden_states_dict:
            raise RuntimeError("Cannot find any *_hidden_states in shift encoder.")

        return hidden_states_dict

    def calculate_layer_wise_loss(self, icv_hidden_states, ice_hidden_states):
        cos_sim = lambda x1, x2: 1 - F.cosine_similarity(x1, x2, dim=-1)
        if Stratety.LAYER_WISE_KL_DIV in self.strategy:
            loss_fn = lambda input, target: F.kl_div(
                input.log_softmax(dim=-1),
                target.softmax(dim=-1),
                reduction="none",
                log_target=False,
            )
        elif Stratety.LAYER_WISE_MSE in self.strategy:
            loss_fn = lambda input, target: F.mse_loss(input, target, reduction="none")
        elif Stratety.LAYER_WISE_COS_SIM in self.strategy:
            loss_fn = cos_sim

        layer_loss = dict()
        for (icv_hs_varname, icv_hs_list), (ice_hs_varname, ice_hs_list) in zip(
            icv_hidden_states.items(), ice_hidden_states.items()
        ):
            # hs_list: batch_size * [num_layer, query_part_len, d_model]

            layer_loss[
                icv_hs_varname.replace(
                    "hidden_states", self.strategy.layer_wise_strategy()
                )
            ] = torch.mean(
                torch.stack(
                    [
                        torch.mean(
                            (1 - cos_sim(icv_hs, ice_hs).unsqueeze(-1))
                            * loss_fn(icv_hs, ice_hs)
                        )
                        for icv_hs, ice_hs in zip(icv_hs_list, ice_hs_list)
                    ]
                )
            )
        return layer_loss

    def calculate_logits_kl_loss(
        self, icv_logits, ice_logits, query_label_inputs, ice_label_mask
    ):
        # extract answer [EOS]
        logits_kl_loss = F.kl_div(
            icv_logits[query_label_inputs].log_softmax(dim=-1),
            ice_logits[ice_label_mask].softmax(dim=-1),
            reduction="batchmean",
            log_target=False,
        )
        return {"logits_kl_loss": logits_kl_loss}

    def forward(self, ice_texts, query_texts, answers, images):
        pad_token, pad_token_id, eos_token = (
            self.lmm.processor.tokenizer.pad_token,
            self.lmm.processor.tokenizer.pad_token_id,
            self.lmm.processor.tokenizer.eos_token,
        )

        hooks = self.shift_encoder.register_record_hooks(self.lmm)

        # step 1. prepare inputs
        full_text = [
            ice + pad_token + query + pad_token + answer + eos_token
            for ice, query, answer in zip(ice_texts, query_texts, answers)
        ]
        inputs = self.lmm.process_input(full_text, images).to(self.device)
        inputs["attention_mask"] = inputs["input_ids"] != pad_token_id

        # step 2. [SOS](implicitly added) ICE [PAD] query [PAD] answer [EOS] forward process
        with torch.no_grad():
            ice_logits = self.lmm.model(**inputs)["logits"]

        # extract query + [PAD] + answer + [EOS]
        ice_hidden_states = (
            self.get_hidden_states(self.generate_label_mask(inputs, 1, keep_bos=True))
            if self.strategy.has_layer_wise()
            else None
        )

        ice_label_mask = self.generate_label_mask(inputs, 2)

        hooks.update(self.shift_encoder.register_shift_hooks(self.lmm))

        loss_dict = {"loss": 0.0}

        # step 1. [SOS](implicitly added) + query + [PAD] + answer [EOS] forward process
        inputs["attention_mask"] = self.generate_label_mask(inputs, 1, keep_bos=True)

        query_outputs = self.lmm.model(
            **inputs,
            labels=inputs["inputs_ids"] if Stratety.LM_LOSS in self.strategy else None,
        )
        icv_logits = query_outputs["logits"]

        if Stratety.LM_LOSS in self.strategy:
            loss_dict["ce_loss"] = query_outputs["loss"]
            loss_dict["loss"] += setting.ce_loss_weight * query_outputs["loss"]

        # extract query + answer + [EOS]
        icv_hidden_states = (
            self.get_hidden_states(inputs["attention_mask"])
            if self.strategy.has_layer_wise()
            else None
        )

        self.remove_hooks(hooks)

        # step 2. calculate kl divergency or MSE of each layer
        if self.strategy.has_layer_wise():
            layer_loss = self.calculate_layer_wise_loss(
                icv_hidden_states, ice_hidden_states
            )
            loss_dict.update(layer_loss)
            loss_dict["loss"] += sum(layer_loss.values())

        # step 3. calculate the last logits kl div
        if Stratety.LOGITS_KL_DIV in self.strategy:
            logits_kl_loss = self.calculate_logits_kl_loss(
                icv_logits,
                ice_logits,
                ice_label_mask,
                ice_label_mask,
            )
            loss_dict.update(logits_kl_loss)
            loss_dict["loss"] += sum(logits_kl_loss.values())

        return loss_dict

    def training_step(self, batch, batch_idx):
        loss_dict = self.forward(**batch)
        self.log_dict(loss_dict, sync_dist=True, prog_bar=True)

        return loss_dict["loss"]

    def configure_optimizers(self):
        param_dict = {
            n: p for n, p in self.shift_encoder.named_parameters() if p.requires_grad
        }
        non_alpha_params = [p for n, p in param_dict.items() if not "alpha" in n]

        optim_groups = [
            {"params": non_alpha_params, "lr": setting.icv_lr},
        ]

        if "deepspeed" in setting.strategy:
            optimizer = DeepSpeedCPUAdam(
                optim_groups,
                weight_decay=setting.weight_decay,
            )
        else:
            optimizer = optim.AdamW(
                optim_groups,
                weight_decay=setting.weight_decay,
            )

        step_batches = self.trainer.estimated_stepping_batches
        warmup_steps = setting.warmup_step
        if isinstance(warmup_steps, float):
            warm_steps = warmup_steps * step_batches
        elif isinstance(warmup_steps, int):
            warm_steps = warmup_steps
        else:
            raise ValueError(
                f"the warm_steps should be int or float, but got {type(warmup_steps)}"
            )
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warm_steps, num_training_steps=step_batches
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }

    def on_save_checkpoint(self, checkpoint):
        checkpoint["state_dict"] = {
            k: v
            for k, v in checkpoint["state_dict"].items()
            if not k.startswith("model")
        }
        return checkpoint

In [24]:
from testbed.models.model_base import HookType
import torch.nn as nn
import re

torch.manual_seed(426)
icv_encoder = AttnFFNShift(
    4096,
    32,
    attn_strategy=ShiftStrategy.USE_VECTOR_IMPL,
    ffn_strategy=ShiftStrategy.RECORD_HIDDEN_STATES,
).to(device, torch.float16)
model = ShiftModel(lmm, icv_encoder, strategy=Stratety.LAYER_WISE_MSE).to(
    device, torch.float16
)

In [25]:
inputs = next(iter(dataloader))
model(**inputs)

{'loss': tensor(0.1379, device='cuda:1', dtype=torch.float16, grad_fn=<AddBackward0>),
 'ffn_mse_loss': tensor(0.1379, device='cuda:1', dtype=torch.float16, grad_fn=<MeanBackward0>)}

In [ ]:
output = model(**inputs)
